<div style="line-height: 1.8;">

# **DefenseFinder → NCBI MG1655**

По результатам анализа с помощью DefenseFinder учли 9 предсказанных защитных систем и 23 белковых компонента. Для этих компонентов оценили соответствие CDS референсной аннотации *E. coli* K-12 MG1655 (`NC_000913.3`).

Использовали подготовленные координаты и последовательности из `results/specialized/defensefinder/`. Сопоставление с NCBI выполнили как отдельную проверку; при сборке итогового выпуска компоненты независимо связали с основным каталогом Bakta.

</div>

In [1]:
from pathlib import Path
import pandas as pd
from Bio import SeqIO

import os

configured_root = os.environ.get("GENOME_ANNOTATION_PROJECT_ROOT")
if configured_root:
    project_dir = Path(configured_root).expanduser().resolve()
else:
    candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "projects/genome_annotation")]
    project_dir = next((p for p in candidates if (p / "data/raw/genome.fna").is_file()), None)
if project_dir is None or not (project_dir / "data/raw/genome.fna").is_file():
    raise FileNotFoundError("Project root not found")
source = project_dir / "results/specialized/defensefinder"
reference = project_dir / "data/processed/NC_000913.3.gbk"

assert source.exists(), f"Missing DefenseFinder directory: {source}"
assert reference.exists(), f"Missing reference: {reference}"

genes = pd.read_csv(source / "defense_genes_with_coordinates.tsv", sep="\t")
systems = pd.read_csv(source / "defense_systems_with_coordinates.tsv", sep="\t")
df_proteins = {r.id: str(r.seq).rstrip('*').upper()
               for r in SeqIO.parse(source / "defensefinder_component_proteins.faa", "fasta")}
reference_record = SeqIO.read(reference, "genbank")

assert len(reference_record.seq) == 4641652
assert len(genes) == 23 and len(systems) == 9
assert genes.hit_id.isin(df_proteins.keys()).all()
print(f"Systems: {len(systems)}; components: {len(genes)}")
print("Reference:", reference_record.id)
input_record = SeqIO.read(project_dir / "data/raw/genome.fna", "fasta")
assert str(reference_record.seq).upper() == str(input_record.seq).upper(), "Reference sequence mismatch"

Systems: 9; components: 23
Reference: NC_000913.3


<div style="line-height: 1.8;">

## **Сопоставление с NCBI**

Перед сравнением координат подтвердили побуквенное совпадение референса с исходным геномом. Для каждого компонента отобрали пересекающиеся CDS на той же цепи и сопоставили их границы и аминокислотные последовательности. Координаты представили в системе с отсчётом от 1 и включением обеих границ.

Кандидатов ранжировали по совпадению координат, белка, стоп-границы и доле перекрытия. Равные оценки лучших кандидатов считали признаком неоднозначности. Совпадение только координат или только белка отделили от одновременного совпадения обоих признаков; одного перекрытия либо общей стоп-границы для автоматической привязки не использовали.

</div>

In [2]:
reference_cds = []
for f in reference_record.features:
    if f.type != "CDS" or f.location.strand not in (1, -1):
        continue
    reference_cds.append({
        "ncbi_start": int(f.location.start) + 1,
        "ncbi_end": int(f.location.end),
        "ncbi_strand": "+" if f.location.strand == 1 else "-",
        "locus_tag": f.qualifiers.get("locus_tag", [None])[0],
        "gene": f.qualifiers.get("gene", [None])[0],
        "product": f.qualifiers.get("product", [None])[0],
        "translation": f.qualifiers.get("translation", [""])[0].rstrip('*').upper(),
    })
ref_cds = pd.DataFrame(reference_cds)

rows = []
for hit in genes.itertuples(index=False):
    aa = df_proteins[hit.hit_id]
    candidates = ref_cds[
        (ref_cds.ncbi_strand == hit.strand) &
        (ref_cds.ncbi_start <= hit.end_1based) &
        (ref_cds.ncbi_end >= hit.start_1based)
    ]
    ranked = []
    for ref in candidates.itertuples(index=False):
        overlap = max(0, min(hit.end_1based, ref.ncbi_end)
                        - max(hit.start_1based, ref.ncbi_start) + 1)
        frac = overlap / (hit.end_1based - hit.start_1based + 1)
        exact_coords = (hit.start_1based == ref.ncbi_start and hit.end_1based == ref.ncbi_end)
        exact_aa = bool(ref.translation) and aa == ref.translation
        same_stop = (hit.end_1based == ref.ncbi_end if hit.strand == "+"
                     else hit.start_1based == ref.ncbi_start)
        rank = (int(exact_coords), int(exact_aa), int(same_stop), frac)
        ranked.append((rank, ref, frac, exact_coords, exact_aa, same_stop))
    ranked.sort(key=lambda x: x[0], reverse=True)
    row = {"sys_id": hit.sys_id, "hit_id": hit.hit_id,
           "defensefinder_model": hit.gene_name,
           "start": hit.start_1based, "end": hit.end_1based,
           "strand": hit.strand, "ncbi_locus_tag": None,
           "ncbi_gene": None, "ncbi_product": None,
           "mapping_status": "no_same_strand_overlap",
           "candidate_locus_tag": None,
           "overlap_fraction": 0.0}
    if ranked:
        rank, ref, frac, exact_coords, exact_aa, same_stop = ranked[0]
        top_tied = (len(ranked)>1 and ranked[1][0]==rank)
        if top_tied:
            status = "ambiguous_tie_review"
        elif exact_coords and exact_aa:
            status = "exact_coords_and_protein"
        elif exact_coords:
            status = "exact_coords_only"
        elif exact_aa:
            status = "exact_protein_only"
        elif same_stop:
            status = "same_stop_review"
        else:
            status = "overlap_review"
        row.update(mapping_status=status, candidate_locus_tag=ref.locus_tag,
                   overlap_fraction=round(frac,4))
        if not top_tied and (exact_aa or exact_coords):
            row.update(ncbi_locus_tag=ref.locus_tag,
                       ncbi_gene=ref.gene, ncbi_product=ref.product)
    rows.append(row)

mapped = pd.DataFrame(rows)
print(mapped.mapping_status.value_counts().to_string())
display(mapped[["hit_id", "defensefinder_model", "ncbi_locus_tag", "ncbi_gene", "mapping_status", "candidate_locus_tag"]])

mapping_status
exact_coords_and_protein    21
same_stop_review             2


,hit_id,defensefinder_model,ncbi_locus_tag,ncbi_gene,mapping_status,candidate_locus_tag
0,1_1111,Lit__Lit,b1139,lit,exact_coords_and_protein,b1139
1,1_1130,RM_Type_IV__Type_IV_05,b1159,mcrA,exact_coords_and_protein,b1159
2,1_2589,Hachiman__HamB,b2627,abpB,exact_coords_and_protein,b2627
3,1_2590,Hachiman__HamA_2,b2628,abpA,exact_coords_and_protein,b2628
4,1_2591,RnlAB__RnlA,b2630,rnlA,exact_coords_and_protein,b2630
5,1_2592,RnlAB__RnlB,b2631,rnlB,exact_coords_and_protein,b2631
6,1_2706,cas2_I-E_2,b2754,cas2,exact_coords_and_protein,b2754
7,1_2707,cas1_I-E_1,b2755,cas1,exact_coords_and_protein,b2755
8,1_2708,cas6e_I_II_III_IV_V_VI_1,b2756,casE,exact_coords_and_protein,b2756
9,1_2709,cas5_I-E_3,b2757,casD,exact_coords_and_protein,b2757


<div style="line-height: 1.8;">

## **Проверка и сохранение**

Результат сохранили в `defensefinder_ncbi_gene_mapping.tsv`. Для записей без однозначной привязки поле `ncbi_locus_tag` оставили незаполненным, а предполагаемый локус указали отдельно.

Установленное соответствие использовали для идентификации компонента в референсной аннотации. Биологическую активность защитных систем на основании этого сопоставления не оценивали.

</div>

In [3]:
outdir = project_dir / "results/specialized/defensefinder"
output_file = outdir / "defensefinder_ncbi_gene_mapping.tsv"
mapped.to_csv(output_file, sep="\t", index=False)

print("Accepted mappings:", mapped.ncbi_locus_tag.notna().sum(), "/", len(mapped))

display(mapped[mapped.ncbi_locus_tag.isna()])

Accepted mappings: 21 / 23


,sys_id,hit_id,defensefinder_model,start,end,strand,ncbi_locus_tag,ncbi_gene,ncbi_product,mapping_status,candidate_locus_tag,overlap_fraction
17,1_MazEF_5,1_4141,MazEF__MazF,4448728,4449042,+,NaN,NaN,NaN,same_stop_review,b4225,1.0000
19,1_RM_Type_IV_10,1_4263,RM_Type_IV__FAM_1,4577958,4579355,-,NaN,NaN,NaN,same_stop_review,b4346,0.9871


<div style="line-height: 1.8;">

Точное совпадение координат и белковой последовательности с NCBI подтвердили для 21 из 23 компонентов. Для `1_4141` и `1_4263` выделили кандидатов `b4225` и `b4346` с общей стоп-границей; эти соответствия сохранили со статусом ручной проверки.

При отдельном сравнении с Bakta точное соответствие установили для всех 23 компонентов. Таким образом, два случая расхождения с NCBI относились к различиям между аннотациями и не нарушили привязку к основному каталогу выпуска.

</div>